# Higher-Order Gaussian Atom-Slice Corrections

This notebook focuses on atomic slice interactions. It plots the atom-plane potential, compares fitted-quadratic Gaussian propagation against a one-slice Fresnel reference after finite propagation, and isolates the single-atom center Taylor failure mode.


## Setup

This diagnostic forces JAX onto CPU because the fitted quadratic action uses many tiny least-squares solves. On some CUDA/JAX installs those tiny solves can fail inside cuSolver. If you already imported JAX in this kernel, restart the kernel and run from the top.


In [ ]:
import os
import sys
import time

if "jax" in sys.modules:
    raise RuntimeError(
        "Restart the kernel and run this notebook from the top. "
        "This notebook sets JAX_PLATFORM_NAME=cpu before importing JAX "
        "to avoid GPU cuSolver failures in small least-squares fits."
    )

os.environ.setdefault("JAX_ENABLE_X64", "1")
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.4")

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from temgym_core.components import AtomicPotential, Detector
from temgym_core.evaluate import evaluate_gaussians_for
from temgym_core.gaussian import (
    FreeSpacePropagator,
    GaussianBeam,
    apply_action_delta,
    make_gaussian,
    taylor_expand,
)
from temgym_core.gaussian_corrections import (
    apply_fitted_quadratic_action,
    concatenate_gaussian_beams,
)
from temgym_core.potential import Si, potential_smoothed_from_r2
from temgym_core.source import square_input_wave
from temgym_core.utils import FresnelPropagator

jax.config.update("jax_enable_x64", True)
print("JAX backend:", jax.devices()[0].platform)


## Shared Helpers

In [ ]:
def aligned_field_error(test, ref):
    alpha = jnp.vdot(test, ref) / jnp.vdot(test, test)
    return float(jnp.linalg.norm(alpha * test - ref) / jnp.linalg.norm(ref))


def normalized_intensity_error(test, ref):
    test_i = jnp.abs(test) ** 2
    ref_i = jnp.abs(ref) ** 2
    test_i = test_i / jnp.sum(test_i)
    ref_i = ref_i / jnp.sum(ref_i)
    return float(jnp.linalg.norm(test_i - ref_i) / jnp.linalg.norm(ref_i))


def multi_atom_action(xy, z, sigma, k, atom_xyz, dz, cutoff_radius):
    def one_atom(atom):
        r2 = (
            (xy[0] - atom[0]) ** 2
            + (xy[1] - atom[1]) ** 2
            + (z - atom[2]) ** 2
        )
        return potential_smoothed_from_r2(r2, Si, cutoff_radius)

    potential = jnp.sum(jax.vmap(one_atom)(atom_xyz))
    return -(sigma / k) * potential * dz


def apply_fitted_slice_to_bundle(
    beam: GaussianBeam,
    atom_xyz,
    dz,
    cutoff_radius,
    *,
    fit_support_radius=1.0,
    fit_samples_per_axis=3,
):
    vector = beam.to_vector()
    out = []
    for i in range(int(np.asarray(vector.x).shape[0])):
        ray = vector[i]
        out.append(
            apply_fitted_quadratic_action(
                ray,
                lambda xy: multi_atom_action(
                    xy, ray.z, ray.sigma, ray.k, atom_xyz, dz, cutoff_radius
                ),
                fit_support_radius=fit_support_radius,
                fit_samples_per_axis=fit_samples_per_axis,
            )
        )
    return concatenate_gaussian_beams(out)


def reference_atom_slice_field(
    input_field, grid, wavelength, atom_xyz, sigma, k, dz, cutoff_radius, propagation
):
    action = jax.vmap(
        lambda xy: multi_atom_action(xy, 0.0, sigma, k, atom_xyz, dz, cutoff_radius)
    )(grid.coords).reshape(grid.shape)
    transmitted = input_field * jnp.exp(1j * k * action)
    return FresnelPropagator(
        transmitted,
        L=grid.pixel_size[0] * grid.shape[0],
        wavelength=wavelength,
        z=propagation,
    )


def run_atom_slice_case(name, beam, grid, atom_xyz, dz, cutoff_radius, propagation):
    t0 = time.time()
    vector = beam.to_vector()
    wavelength = vector.wavelength[0]
    sigma = vector.sigma[0]
    k = vector.k[0]

    input_field = evaluate_gaussians_for(beam, grid)
    reference = reference_atom_slice_field(
        input_field, grid, wavelength, atom_xyz, sigma, k, dz, cutoff_radius, propagation
    )
    fitted = apply_fitted_slice_to_bundle(beam, atom_xyz, dz, cutoff_radius)
    propagated = FreeSpacePropagator()(fitted, propagation)
    gaussian = evaluate_gaussians_for(propagated, grid)

    result = {
        "name": name,
        "beams": int(np.asarray(vector.x).size),
        "field_error": aligned_field_error(gaussian, reference),
        "intensity_error": normalized_intensity_error(gaussian, reference),
        "seconds": time.time() - t0,
        "input": input_field,
        "reference": reference,
        "gaussian": gaussian,
    }
    print(
        f"{name}: beams={result['beams']}, "
        f"field_error={result['field_error']:.4g}, "
        f"intensity_error={result['intensity_error']:.4g}, "
        f"time={result['seconds']:.1f}s"
    )
    return result


def show_case(result, extent):
    reference = np.asarray(result["reference"])
    gaussian = np.asarray(result["gaussian"])
    diff = np.abs(gaussian) ** 2 - np.abs(reference) ** 2

    fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex=True, sharey=True)
    images = [
        (np.abs(reference), "Reference amplitude", "inferno"),
        (np.abs(gaussian), "Gaussian amplitude", "inferno"),
        (diff, "Intensity difference", "coolwarm"),
        (np.angle(reference), "Reference phase", "twilight"),
        (np.angle(gaussian), "Gaussian phase", "twilight"),
        (np.angle(gaussian * np.conj(reference)), "Relative phase", "twilight"),
    ]
    for ax, (data, title, cmap) in zip(axes.ravel(), images):
        im = ax.imshow(data, extent=extent, origin="lower", cmap=cmap)
        ax.set_title(title)
        fig.colorbar(im, ax=ax, shrink=0.75)
    fig.suptitle(
        f"{result['name']} | field error={result['field_error']:.3g}, "
        f"intensity error={result['intensity_error']:.3g}"
    )
    fig.tight_layout()
    return fig

## 1. Plane Of Atoms: Fitted Gaussian Slice vs Grid Reference

This is the main runnable diagnostic. The reference is a one-slice grid transmission followed by Fresnel propagation. The Gaussian path fits a local quadratic action over each beam support, applies it analytically, and then propagates the Gaussian bundle.

In [ ]:
voltage = 100e3
extent = 4.0  # angstrom half-width
n_px = 48
dz = 0.05  # angstrom slice thickness
propagation = 2.0  # angstrom free-space propagation after the slice
pixel = 2 * extent / n_px
cutoff_radius = pixel / 3.0

grid = Detector(z=0.0, pixel_size=(pixel, pixel), shape=(n_px, n_px))
plot_extent = (grid.extent[0], grid.extent[1], grid.extent[2], grid.extent[3])

# A 2x2 plane of atoms. Si is used because the repo currently includes Si parameters.
spacing = 2.35
atom_xyz = jnp.array(
    [
        [-0.5 * spacing, -0.5 * spacing, 0.0],
        [0.5 * spacing, -0.5 * spacing, 0.0],
        [-0.5 * spacing, 0.5 * spacing, 0.0],
        [0.5 * spacing, 0.5 * spacing, 0.0],
    ]
)

tem_beam = square_input_wave(
    aperture_length=2 * extent,
    waist=2.0,
    voltage=voltage,
    overlap_factor=1.0,
    wavelength_unit="angstrom",
)
stem_probe = make_gaussian(
    x=0.0,
    y=0.0,
    z=0.0,
    voltage=voltage,
    waist_x=0.5,
    waist_y=0.5,
    wavelength_unit="angstrom",
)

atom_results = {
    "TEM": run_atom_slice_case(
        "TEM-like broad beam", tem_beam, grid, atom_xyz, dz, cutoff_radius, propagation
    ),
    "STEM": run_atom_slice_case(
        "STEM-like focused probe", stem_probe, grid, atom_xyz, dz, cutoff_radius, propagation
    ),
}

In [ ]:
show_case(atom_results["TEM"], plot_extent)
show_case(atom_results["STEM"], plot_extent)
plt.show()

## Atom Potential And Thin-Slice Phase

This shows the scalar potential used for the 2x2 atom plane and the corresponding thin-slice phase shift for the TEM/STEM comparison above.

In [ ]:
def atom_plane_potential(coords, atom_xyz, cutoff_radius):
    def value_at_xy(xy):
        def one_atom(atom):
            r2 = (xy[0] - atom[0]) ** 2 + (xy[1] - atom[1]) ** 2 + atom[2] ** 2
            return potential_smoothed_from_r2(r2, Si, cutoff_radius)

        return jnp.sum(jax.vmap(one_atom)(atom_xyz))

    return jax.vmap(value_at_xy)(coords).reshape(grid.shape)


potential_map = atom_plane_potential(grid.coords, atom_xyz, cutoff_radius)
tem_vector = tem_beam.to_vector()
slice_action = jax.vmap(
    lambda xy: multi_atom_action(
        xy,
        0.0,
        tem_vector.sigma[0],
        tem_vector.k[0],
        atom_xyz,
        dz,
        cutoff_radius,
    )
)(grid.coords).reshape(grid.shape)
phase_map = tem_vector.k[0] * slice_action

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
im0 = axes[0].imshow(np.asarray(potential_map), extent=plot_extent, origin="lower")
axes[0].plot(np.asarray(atom_xyz[:, 0]), np.asarray(atom_xyz[:, 1]), "wo", ms=4, mec="k")
axes[0].set_title("Projected atom-plane potential")
axes[0].set_xlabel("x (A)")
axes[0].set_ylabel("y (A)")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(np.asarray(phase_map), extent=plot_extent, origin="lower", cmap="twilight")
axes[1].plot(np.asarray(atom_xyz[:, 0]), np.asarray(atom_xyz[:, 1]), "wo", ms=4, mec="k")
axes[1].set_title(f"Thin-slice phase for dz={dz} A")
axes[1].set_xlabel("x (A)")
fig.colorbar(im1, ax=axes[1], shrink=0.8, label="rad")

fig.tight_layout()
plt.show()

## Propagated Gaussian Fields vs Fresnel Reference

These line profiles are taken after the fitted Gaussian slice interaction and the same free-space propagation distance used by the Fresnel reference. This is the direct check that the Gaussian forward model can track the atom-slice interaction after propagation.

In [ ]:
x_coords = np.asarray(grid.coords_1d[0])
center_y = grid.shape[0] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
for row, key in enumerate(["TEM", "STEM"]):
    result = atom_results[key]
    reference = np.asarray(result["reference"])
    gaussian = np.asarray(result["gaussian"])
    reference_intensity = np.abs(reference[center_y, :]) ** 2
    gaussian_intensity = np.abs(gaussian[center_y, :]) ** 2

    axes[row, 0].plot(x_coords, np.abs(reference[center_y, :]), label="Fresnel reference")
    axes[row, 0].plot(x_coords, np.abs(gaussian[center_y, :]), "--", label="Gaussian fitted")
    axes[row, 0].set_ylabel(f"{key}\namplitude")

    axes[row, 1].plot(x_coords, reference_intensity, label="Fresnel reference")
    axes[row, 1].plot(x_coords, gaussian_intensity, "--", label="Gaussian fitted")
    axes[row, 1].set_ylabel("intensity")

    relative_phase = np.angle(gaussian[center_y, :] * np.conj(reference[center_y, :]))
    axes[row, 2].plot(x_coords, relative_phase)
    axes[row, 2].set_ylabel("relative phase (rad)")

for ax in axes[-1, :]:
    ax.set_xlabel("x (A)")
for ax in axes.ravel()[:2]:
    ax.legend()

fig.suptitle(f"After atom slice and {propagation} A propagation")
fig.tight_layout()
plt.show()

## 2. Single Atom: Center Taylor vs Fitted Quadratic

This cell isolates the important atom-core effect. A Gaussian centered exactly on the atom is a bad case for derivative Taylor expansion, but sampled fitted quadratics handle it well.

In [ ]:
single_extent = 3.0
single_n = 128
single_pixel = 2 * single_extent / single_n
single_grid = Detector(
    z=0.0,
    pixel_size=(single_pixel, single_pixel),
    shape=(single_n, single_n),
)
single_atom = AtomicPotential(
    atom_xyz=jnp.array([0.0, 0.0, 0.0]),
    element_params=Si,
    cutoff_radius=single_pixel / 3.0,
    z=0.0,
)
single_ray = make_gaussian(
    x=0.0,
    y=0.0,
    z=0.0,
    voltage=100e3,
    waist_x=0.5,
    waist_y=0.5,
    wavelength_unit="angstrom",
)
single_dz = 0.05


def taylor_atom_kick(ray, atom, dz):
    dS0, dS1, dS2 = taylor_expand(atom.complex_action, ray.r_xy, ray.z, ray.sigma, ray.k)
    r_xy, d_xy, amplitude, pathlength, q_inv = apply_action_delta(
        ray, dS0=dS0 * dz, dS1=dS1 * dz, dS2=dS2 * dz
    )
    return ray.derive(
        x=r_xy[0],
        y=r_xy[1],
        dx=d_xy[0],
        dy=d_xy[1],
        amplitude=amplitude,
        pathlength=pathlength,
        Q_inv=q_inv,
    )


sigma = single_ray.sigma
k = single_ray.k
phase_grid = jax.vmap(single_atom.phase_shift, in_axes=(0, None, None, None))(
    single_grid.coords, 0.0, sigma, k
).reshape(single_grid.shape)
single_input = evaluate_gaussians_for(single_ray, single_grid)
single_reference = single_input * jnp.exp(1j * k * phase_grid * single_dz)

single_taylor = evaluate_gaussians_for(
    taylor_atom_kick(single_ray, single_atom, single_dz), single_grid
)
single_fitted_ray = apply_fitted_quadratic_action(
    single_ray,
    lambda xy: single_atom.complex_action(xy, single_ray.z, single_ray.sigma, single_ray.k)
    * single_dz,
    fit_support_radius=1.0,
    fit_samples_per_axis=7,
)
single_fitted = evaluate_gaussians_for(single_fitted_ray, single_grid)

print("Taylor field error:", aligned_field_error(single_taylor, single_reference))
print("Fitted field error:", aligned_field_error(single_fitted, single_reference))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharex=True, sharey=True)
for ax, field, title in [
    (axes[0], single_reference, "Exact grid transmission"),
    (axes[1], single_taylor, "Center Taylor Gaussian"),
    (axes[2], single_fitted, "Fitted quadratic Gaussian"),
]:
    im = ax.imshow(np.abs(field), extent=single_grid.extent, origin="lower", cmap="inferno")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.75)
fig.tight_layout()
plt.show()